<a href="https://githubtocolab.com/eskayML/issr_gsoc2026_communication_analysis/blob/main/01_Data_Resource_Selection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install dependencies if running in Colab
import sys
if 'google.colab' in sys.modules:
    !pip install librosa==0.11.0 noisereduce==3.0.3 soundfile==0.12.1 matplotlib==3.7.1

# GSoC 2026: ISSR Project 01 Test
## Notebook 1: Data Resource Selection and Sample Evaluation

**Applicant:** Samuel Kalu  
**Organization:** Institute for Social Science Research (ISSR)

> [!NOTE]
> This test is a continuation of my commitment to the ISSR lab's research goals. My **GSoC 2025** implementation ([read the final report here](https://eskayml.medium.com/gsoc-25-communication-analysis-tool-for-human-ai-interaction-driving-simulator-experiments-final-39c53d90672d)) laid the groundwork for large-scale simulator audio analysis. This 2026 iteration aims to refine the data ingestion and enhancement stages for even higher-density interactions.

### 1. Database Identification: The AMI Meeting Corpus
To address the project's requirement for a team communication environment, I have identified the **AMI Meeting Corpus** as the optimal data resource.

- **Multimodal Data:** It features synchronized video, close-talk headset microphones, and circular microphone arrays (8-channel circular).
- **Environment:** It captures 100 hours of meetings involving teams simulating a collaborative design process.
- **Human-Factors Focus:** Unlike many datasets that focus on pure transcription, the AMI corpus includes rich annotations for social signals, which aligns with how ISSR analyzes human-AI interaction.

### 2. My Perspective: Why AMI for the ISSR Driving Simulator?

#### a. Scaling to the 6+1 Participant Structure
Having worked with the ISSR lab's footage previously, I know that our driving simulator environment is uniquely challenging because it involves **6 participants plus a narrator** (7 total) communicating simultaneously. Most open-source meeting datasets only feature 2 or 3 people, which doesn't capture the density of our data. 

I chose the AMI Corpus because its 8-channel microphone arrays are the only open-source proxy capable of simulating the complex spatial and acoustic challenges we face when 7 speakers are active in the same room. 

#### b. How this data will be used?
We will use the **AMI Microphone Array recordings** to simulate the distant microphone setup of a simulator and inject artificial driving noise. The high-fidelity headset microphones from AMI will serve as the 'Clean Reference' to measure exactly how much clarity our enhancement pipeline adds to the original signal.


In [ ]:
import os
import librosa
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
import librosa.display
import requests
from tqdm import tqdm

def download_ami_sample(target_path, url):
    """
    Downloads an actual 16kHz audio sample from the AMI Meeting Corpus repository.
    """
    if os.path.exists(target_path):
        print(f"Sample already exists at {target_path}")
        return

    print(f"Downloading AMI Sample (ES2008a) from University of Edinburgh...")
    os.makedirs(os.path.dirname(target_path), exist_ok=True)
    
    response = requests.get(url, stream=True)
    total_size = int(response.headers.get('content-length', 0))
    
    with open(target_path, 'wb') as file, tqdm(
        desc=target_path,
        total=total_size,
        unit='iB',
        unit_scale=True,
        unit_divisor=1024,
    ) as bar:
        for data in response.iter_content(chunk_size=1024):
            size = file.write(data)
            bar.update(size)

sample_url = "http://groups.inf.ed.ac.uk/ami/AMICorpusMirror/signals/headsetmix/ES2008a.mix.wav"
sample_path = "input/AMI_sample.wav"

download_ami_sample(sample_path, sample_url)

y, sr = librosa.load(sample_path, sr=16000)
print(f"Sample ready at: {sample_path}")
print(f"Audio Duration: {librosa.get_duration(y=y, sr=sr):.2f} seconds")
print(f"Sample Rate: {sr} Hz")


### 3. Exploratory Data Analysis (EDA)
Visualizing the spectral properties of the actual team communication sample to understand the baseline signal quality.


In [ ]:
plt.figure(figsize=(14, 8))

# Plot Waveform (subset for visibility)
plt.subplot(2, 1, 1)
librosa.display.waveshow(y[:160000], sr=sr, color='blue') # First 10 seconds
plt.title('Waveform of Team Communication Sample (Actual AMI ES2008a)')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')

# Plot Spectrogram
plt.subplot(2, 1, 2)
D = librosa.amplitude_to_db(np.abs(librosa.stft(y[:160000])), ref=np.max)
librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='hz')
plt.colorbar(format='%+2.0f dB')
plt.title('Spectrogram (Speech Formant Distribution)')

plt.tight_layout()
plt.show()
